# Latent Space Navigation

Navigate the model's latent space: interpolation, arithmetic,
boundary detection, manifold exploration, and distance mapping.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.latent_space_navigation import (
    representation_interpolation,
    latent_arithmetic,
    boundary_detection,
    manifold_exploration,
    latent_distance_map,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens_a = jnp.array([1, 2, 3, 4, 5, 6, 7])
tokens_b = jnp.array([50, 51, 52, 53, 54, 55, 56])
tokens_c = jnp.array([10, 20, 30, 40, 50, 60, 70])
print("Model ready")

## Representation Interpolation

Linear interpolation between two activations, observing prediction changes.

In [ ]:
result = representation_interpolation(model, tokens_a, tokens_b, n_steps=8)
print(f"Path length: {result['total_path_length']:.4f}")
print(f"\nInterpolation trajectory:")
for i, (alpha, pred, ent) in enumerate(zip(
    result['alphas'], result['top_predictions'], result['prediction_entropy'])):
    print(f"  α={float(alpha):.2f}: pred={pred}, entropy={float(ent):.4f}")

## Latent Arithmetic

Vector arithmetic: a - b + c in activation space.

In [ ]:
result = latent_arithmetic(model, tokens_a, tokens_b, tokens_c)
print(f"Top predictions: {result['top_predictions'][:3]}")
print(f"Cosine to a: {result['cosine_to_a']:.4f}")
print(f"Cosine to b: {result['cosine_to_b']:.4f}")
print(f"Cosine to c: {result['cosine_to_c']:.4f}")
print(f"Result norm: {result['result_norm']:.4f}")

## Boundary Detection

Find where the model's top prediction changes along an interpolation.

In [ ]:
result = boundary_detection(model, tokens_a, tokens_b, n_probes=15)
print(f"Number of boundaries: {result['n_boundaries']}")
print(f"Boundary positions (α): {result['boundary_alphas']}")
print(f"Prediction sequence: {result['prediction_sequence']}")
print(f"Boundary sharpness: {result['boundary_sharpness']:.4f}")

## Manifold Exploration

Explore the local activation manifold around a point.

In [ ]:
result = manifold_exploration(model, tokens_a, n_directions=4, n_steps=3)
print(f"Center prediction: {result['center_prediction']}")
print(f"Prediction stability: {result['prediction_stability']:.1%}")
print(f"Prediction changes: {result['n_prediction_changes']}/{result['total_probes']}")

## Latent Distance Map

In [ ]:
tokens_list = [tokens_a, tokens_b, tokens_c,
               jnp.array([5, 10, 15, 20, 25, 30, 35])]
result = latent_distance_map(model, tokens_list)
print(f"Mean distance: {result['mean_distance']:.4f}")
print(f"Mean cosine: {result['mean_cosine']:.4f}")
print(f"\nNearest neighbors:")
for i, nn, dist in result['nearest_neighbors']:
    print(f"  Input {i} -> Input {nn} (dist={dist:.4f})")